# Practical 14

**Aim:** Demonstrate basic machine learning workflow using Scikit-learn: data loading, splitting, model training and evaluation.

**Dataset:** [Wine Quality Dataset - UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/186/wine+quality) (Loaded directly via `ucimlrepo` module)

### Step 1: Data Loading via `ucimlrepo` Module

In [1]:
from ucimlrepo import fetch_ucirepo

# Fetch Wine Quality dataset directly via official UCI repository module (id=186)
wine_quality = fetch_ucirepo(id=186)

# Access features and targets as pandas DataFrames
X = wine_quality.data.features
y = wine_quality.data.targets

print("Features Shape (Rows, Columns):", X.shape)
print("Targets Shape  (Rows, Columns):", y.shape)
print("\nFirst 5 Records of Features:")
X.head()

Features Shape (Rows, Columns): (6497, 11)
Targets Shape  (Rows, Columns): (6497, 1)

First 5 Records of Features:


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4


### Step 2: Data Cleaning

In [2]:
import pandas as pd

# 1. Check for missing/null values across features
print("Missing values in features:")
print(X.isnull().sum())

# 2. Check and remove duplicate rows
data_combined = pd.concat([X, y], axis=1)
initial_count = len(data_combined)

# Drop missing values (if any) and duplicate rows
data_cleaned = data_combined.dropna().drop_duplicates().reset_index(drop=True)
cleaned_count = len(data_cleaned)

print(f"\nInitial records        : {initial_count}")
print(f"Duplicate rows dropped : {initial_count - cleaned_count}")
print(f"Cleaned records        : {cleaned_count}")

# 3. Update X and y with cleaned dataset
X = data_cleaned.drop(columns=["quality"])
y = data_cleaned[["quality"]]

Missing values in features:
fixed_acidity           0
volatile_acidity        0
citric_acid             0
residual_sugar          0
chlorides               0
free_sulfur_dioxide     0
total_sulfur_dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
dtype: int64

Initial records        : 6497
Duplicate rows dropped : 1179
Cleaned records        : 5318


### Step 3: Data Splitting (Features, Target & Train-Test Split)

In [3]:
from sklearn.model_selection import train_test_split

# Convert target 'quality' into binary classification:
# 1 = Good Quality (rating >= 6), 0 = Average/Bad Quality (rating < 6)
y_binary = (y["quality"] >= 6).astype(int)

# Split dataset into 80% training and 20% testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42
)

print("Total Cleaned Dataset Size :", len(X))
print("Training Set Size          :", X_train.shape[0])
print("Testing Set Size           :", X_test.shape[0])
print("Number of Features         :", X_train.shape[1])

Total Cleaned Dataset Size : 5318
Training Set Size          : 4254
Testing Set Size           : 1064
Number of Features         : 11


### Step 4: Model Training using Scikit-learn

In [4]:
from sklearn.ensemble import RandomForestClassifier

# Initialize and train the Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Model Training Completed Successfully!")
print("Trained Model:", model)

Model Training Completed Successfully!
Trained Model: RandomForestClassifier(random_state=42)


### Step 5: Model Evaluation

In [5]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Predict on test data
y_pred = model.predict(X_test)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(
    y_test, y_pred, target_names=["Average/Bad (0)", "Good (1)"]
)

print(f"Model Accuracy: {accuracy * 100:.2f}%\n")
print("Confusion Matrix:")
print(conf_matrix)
print("\nClassification Report:")
print(class_report)

Model Accuracy: 76.60%

Confusion Matrix:
[[259 124]
 [125 556]]

Classification Report:
                 precision    recall  f1-score   support

Average/Bad (0)       0.67      0.68      0.68       383
       Good (1)       0.82      0.82      0.82       681

       accuracy                           0.77      1064
      macro avg       0.75      0.75      0.75      1064
   weighted avg       0.77      0.77      0.77      1064



### Step 6: Sample Prediction on Unseen Data

In [6]:
# Demonstrate inference on a sample wine instance
sample_wine = X_test.iloc[[0]]
actual_label = y_test.iloc[0]
predicted_label = model.predict(sample_wine)[0]

print("Sample Wine Features:\n", sample_wine)
print(f"\nActual Quality    : {'Good (1)' if actual_label == 1 else 'Average/Bad (0)'}")
print(f"Predicted Quality : {'Good (1)' if predicted_label == 1 else 'Average/Bad (0)'}")

Sample Wine Features:
       fixed_acidity  volatile_acidity  citric_acid  ...    pH  sulphates  alcohol
4697            6.6              0.23         0.32  ...  3.22       0.39     12.7

[1 rows x 11 columns]

Actual Quality    : Good (1)
Predicted Quality : Good (1)
